1 Cài thư viện
--------------

In [2]:
!pip install transformers accelerate -q

2 Import thư viện
-----------------

In [3]:
import pandas as pd
import torch
import re
from transformers import AutoTokenizer, AutoModelForCausalLM
from tqdm import tqdm
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

3 Load model Qwen ( Instruct)
-----------------

In [4]:
model_name = "/kaggle/input/models/marquis03/qwen2/transformers/qwen2-7b-instruct/1"

tokenizer = AutoTokenizer.from_pretrained(model_name)

tokenizer.padding_side = 'left'

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float16,
    device_map="auto"
)

model.eval()

`torch_dtype` is deprecated! Use `dtype` instead!


Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(152064, 3584)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=3584, out_features=3584, bias=True)
          (k_proj): Linear(in_features=3584, out_features=512, bias=True)
          (v_proj): Linear(in_features=3584, out_features=512, bias=True)
          (o_proj): Linear(in_features=3584, out_features=3584, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=3584, out_features=18944, bias=False)
          (up_proj): Linear(in_features=3584, out_features=18944, bias=False)
          (down_proj): Linear(in_features=18944, out_features=3584, bias=False)
          (act_fn): SiLUActivation()
        )
        (input_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((3584,), eps=1e-06)
    (ro

4 Load Dataset
--------------

In [5]:
df = pd.read_csv("/kaggle/input/datasets/giba1010/multilexnorm-2026/validation_vi.csv")

# Chuyển norm từ list dạng string -> string thực sự
def parse_norm_column(x):
    # Nếu đã là string thì xử lý
    if isinstance(x, str):
        # Loại bỏ dấu ngoặc [], thay dấu ',' bằng space
        x = x.strip("[]")
        # Tách các phần tử trong list
        import ast
        try:
            # Thử parse như list Python
            items = ast.literal_eval(f"[{x}]") if not x.startswith('[') else ast.literal_eval(x)
            return ' '.join(items)
        except:
            # Nếu lỗi thì thay thế đơn giản
            x = re.sub(r"['\"]", "", x)
            x = re.sub(r",", " ", x)
            return ' '.join(x.split())
    return str(x)

df['norm_str'] = df['norm'].apply(parse_norm_column)
df['raw_str'] = df['raw'].apply(lambda x: ' '.join(x) if isinstance(x, list) else str(x))

texts = df["raw_str"].astype(str).tolist()

print(df[['raw_str', 'norm_str']].head())

                                             raw_str  \
0  ['Bên' 'ni' 'đèo' 'thì' '"bả' 'sá",' 'bên' 'tê...   
1  ['2' 'mày' 'bán' 'tao' 'ngồi' 'đếm' 'xiền' 'xe...   
2  ['Vứt' 'con' 'đấy' 'đi' 'hành' 'hung' 'ng' 'kh...   
3  ['V' 'là' 'ai' 'ở' 'nhà' 'ôm' 'con' 'cũng' 'rả...   
4                       ['3' 'đứa' 'm' 'có' 'v' 'k']   

                                            norm_str  
0              Bênnàyđèothì"bảsá",bênkiađèothì"bẻbẻ"  
1  2màybántaongồiđếmtiềnxemcóđủcho3đứahọclạikhông...  
2  Vứtconđấyđihànhhungngườikháctrongkhibảnthânsai...  
3                 Vậylàaiởnhàômconcũngrảnhhánghếthả?  
4                                    3đứamcóvậykhông  


In [33]:
print(df.head())

                                                 raw  \
0  ['Bên' 'ni' 'đèo' 'thì' '"bả' 'sá",' 'bên' 'tê...   
1  ['2' 'mày' 'bán' 'tao' 'ngồi' 'đếm' 'xiền' 'xe...   
2  ['Vứt' 'con' 'đấy' 'đi' 'hành' 'hung' 'ng' 'kh...   
3  ['V' 'là' 'ai' 'ở' 'nhà' 'ôm' 'con' 'cũng' 'rả...   
4                       ['3' 'đứa' 'm' 'có' 'v' 'k']   

                                                norm lang       split  \
0  ['Bên' 'này' 'đèo' 'thì' '"bả' 'sá",' 'bên' 'k...   vi  validation   
1  ['2' 'mày' 'bán' 'tao' 'ngồi' 'đếm' 'tiền' 'xe...   vi  validation   
2  ['Vứt' 'con' 'đấy' 'đi' 'hành' 'hung' 'người' ...   vi  validation   
3  ['Vậy' 'là' 'ai' 'ở' 'nhà' 'ôm' 'con' 'cũng' '...   vi  validation   
4                 ['3' 'đứa' 'm' 'có' 'vậy' 'không']   vi  validation   

                                            norm_str  \
0              Bênnàyđèothì"bảsá",bênkiađèothì"bẻbẻ"   
1  2màybántaongồiđếmtiềnxemcóđủcho3đứahọclạikhông...   
2  Vứtconđấyđihànhhungngườikháctrongkhibảnthânsai...   


5 Normalize (Không finetune)
------------------------

In [21]:
def normalize_batch(texts):
    results = []
    
    for text in texts:
        # Xử lý text đầu vào
        if isinstance(text, list):
            text = ' '.join(text)
        elif isinstance(text, str) and text.startswith("['"):
            try:
                import ast
                text_list = ast.literal_eval(text)
                text = ' '.join(text_list)
            except:
                pass
        
        # Prompt (đã fix)
        prompt = f"""Nhiệm vụ: Chuẩn hóa văn bản.

Quy tắc:
- Giữ nguyên nghĩa và số lượng từ
- Không thêm, không bớt nội dung
- Chỉ sửa lỗi chính tả và viết tắt (k -> không, t -> tôi, m -> mày, v -> vậy)
- Không diễn đạt lại câu

Ví dụ:
Input: k dc
Output: không được

Input: t ko biet
Output: tôi không biết

Input: {text}
Output:"""
        
        inputs = tokenizer(
            prompt,
            return_tensors="pt",
            truncation=True,
            max_length=512
        ).to(model.device)
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=50,
                temperature=0.0,
                do_sample=False,
                repetition_penalty=1.1,
                no_repeat_ngram_size=3,
                pad_token_id=tokenizer.eos_token_id
            )
        
        # Lấy phần generate (không dính prompt)
        output_ids = outputs[0][inputs['input_ids'].shape[1]:]
        decoded = tokenizer.decode(output_ids, skip_special_tokens=True)
        
        result = decoded.strip().split('\n')[0]
        result = result.strip('"\'')
        
        # match format dataset (không space)
        result = result.replace(" ", "")
        
        results.append(result)
    
    return results

6 Hàm inference
---------------

In [22]:
def normalize_batch(texts, batch_size=8):
    results = []
    
    for i in range(0, len(texts), batch_size):
        batch = texts[i:i+batch_size]
        
        processed_batch = []
        for text in batch:
            if isinstance(text, list):
                text = ' '.join(text)
            processed_batch.append(text)
        
        prompts = [f"""Chuẩn hóa văn bản, không thêm bớt:

Input: {t}
Output:""" for t in processed_batch]
        
        inputs = tokenizer(
            prompts,
            return_tensors="pt",
            padding=True,
            truncation=True,
            max_length=256
        ).to(model.device)
        
        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=40,
                do_sample=False,
                temperature=0.0,
                pad_token_id=tokenizer.eos_token_id
            )
        
        for j in range(len(batch)):
            output_ids = outputs[j][inputs['input_ids'].shape[1]:]
            decoded = tokenizer.decode(output_ids, skip_special_tokens=True)
            result = decoded.strip().split('\n')[0]
            result = result.replace(" ", "")
            results.append(result)
    
    return results

7 Chạy inference
----------------

In [23]:
batch_size = 8
preds = []

for i in tqdm(range(0, len(texts), batch_size)):
    batch = texts[i:i+batch_size]
    preds.extend(normalize_batch(batch))

df["pred"] = preds

100%|██████████| 132/132 [08:09<00:00,  3.71s/it]


7 Evaluate
----------

In [24]:
def clean_text(x):
    x = str(x).lower().strip()
    x = re.sub(r'\s+', ' ', x)
    return x

df_eval = df.copy()
df_eval["pred_clean"] = df_eval["pred"].apply(clean_text)
df_eval["norm_clean"] = df_eval["norm_str"].apply(clean_text)

# In sample để kiểm tra
for i in range(10):
    print(f"{i+1}. PRED: {df_eval.loc[i, 'pred_clean'][:50]}")
    print(f"   NORM: {df_eval.loc[i, 'norm_clean'][:50]}")
    print(f"   MATCH: {df_eval.loc[i, 'pred_clean'] == df_eval.loc[i, 'norm_clean']}")

# Tính accuracy đơn giản trước
correct = (df_eval["pred_clean"] == df_eval["norm_clean"]).sum()
total = len(df_eval)
simple_acc = correct / total

print(f"\nACCURACY ĐƠN GIẢN (so sánh chính xác): {simple_acc:.4f} ({correct}/{total})")

1. PRED: ['bên''nị''đèo''thì''"bả''sá','bên''tệ''đèo''th
   NORM: bênnàyđèothì"bảsá",bênkiađèothì"bẻbẻ"
   MATCH: False
2. PRED: "2màybántaongồiđếmxiềnxemcóđủcho3đứahọclạikhôngnhà
   NORM: 2màybántaongồiđếmtiềnxemcóđủcho3đứahọclạikhôngnhan
   MATCH: False
3. PRED: "vứtconđóđihànhhungngkháctrongkhibảnthânsailèra."
   NORM: vứtconđấyđihànhhungngườikháctrongkhibảnthânsailèra
   MATCH: False
4. PRED: "vlàaiởnhàômconcũngrảnhhánghếthả?"
   NORM: vậylàaiởnhàômconcũngrảnhhánghếthả?
   MATCH: False
5. PRED: ['3','đứa','m','có','v','k']
   NORM: 3đứamcóvậykhông
   MATCH: False
6. PRED: "ngonvàohốcthoảimáikhôngphảikiêngdèai=))"
   NORM: ngon,vàohốcthoảimáikhôngphảikiêngdèai=))
   MATCH: False
7. PRED: "giờnghĩlạithấylớp6taimìnhcũngkhỏe"
   NORM: giờnghĩlạithấylớp6taimìnhcũngkhỏethiệt
   MATCH: False
8. PRED: "đọctiêuđềbachấmwua"
   NORM: đọctiêuđềbachấmquá
   MATCH: False
9. PRED: ['thành','ra','đến','hyvong','đương','vẫn','phải',
   NORM: thànhrađếngiờyêuđươngvẫnphảigiấudiếmchứchảđượcthoả
   MAT